# Supporting Experiments for the Codec-Degradation Study
## Speech-vs-Environmental Contrast, Augmentation Baseline, and ESC-50 Generalisation

This notebook runs three experiments that strengthen the main codec-degradation paper. All share the protocol of the main experiment (ResNet-50, mel-spectrogram input, fixed seed 42, cross-validation over predefined folds) so results are directly comparable.

**Experiment 1 -- Speech versus environmental contrast.** Keyword classification on Speech Commands is degraded through the same AMR-NB 4.75 kbit/s pipeline used for environmental sound. If speech degrades far less than environmental sound under identical coding, the source-model-mismatch account is demonstrated at the task level, not only acoustically.

**Experiment 2 -- Codec-agnostic augmentation baseline.** A model trained with generic augmentation (additive noise and spectrogram masking, no codec) is tested on AMR-NB audio. Comparing its recovery against codec-aware augmentation isolates the value of matching the augmentation to the deployment codec.

**Experiment 3 -- ESC-50 generalisation.** The full clean/matched/augmented pipeline is repeated on ESC-50, a second environmental corpus with 50 classes, testing whether the findings generalise beyond UrbanSound8K and enabling a per-class frequency-dependence analysis at higher class count.

**Runtime:** approximately seven hours on a T4 GPU; every fold is checkpointed, so an interrupted session resumes without recomputation.

**Setup:** GPU T4, Internet On, and add `gurjant-us8k-official`, `gurjant-esc50-official`, and `gurjant-sc_v2-official` via + Add Input.

In [1]:
# Cell 1 - Configuration
import os, sys, json, random, subprocess, csv, warnings
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore", message="n_fft=.* is too large")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

IN      = Path("/kaggle/input")
WORK    = Path("/kaggle/working")
RESULTS = WORK / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
CKPT    = RESULTS / "ckpt"; CKPT.mkdir(exist_ok=True)
PROC    = WORK / "processed"; PROC.mkdir(exist_ok=True)
MEL     = WORK / "mel_cache"; MEL.mkdir(exist_ok=True)
TMP     = WORK / "tmp"; TMP.mkdir(exist_ok=True)

TARGET_SR = 22050; MAX_DUR = 4.0
N_MELS = 128; N_FFT = 2048; HOP = 512
BATCH = 32; EPOCHS = 30; PATIENCE = 5; LR = 1e-4
AMR_KBPS = "4.75k"

# Reference values from the main experiment for context in the analysis cells
REF_US8K_CLEAN = 0.786
REF_US8K_AMR   = 0.370
REF_US8K_AUG   = 0.698   # codec-aware augmentation at 4.75k

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | CUDA devices: {torch.cuda.device_count()}")
assert DEVICE.type == "cuda", "Enable GPU T4 in Settings before running."
print("Configuration loaded.")

Device: cuda | CUDA devices: 2
Configuration loaded.


In [2]:
# Cell 2 - Install AMR-NB codec and dependencies
subprocess.run("apt-get update -qq && apt-get install -y -qq "
               "libavcodec-extra libopencore-amrnb0 libopencore-amrwb0 ffmpeg",
               shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
enc = subprocess.run(["ffmpeg","-encoders"], capture_output=True, text=True).stdout
if "libopencore_amrnb" not in enc:
    subprocess.run("wget -q https://johnvansickle.com/ffmpeg/releases/"
                   "ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz",
                   shell=True)
    st = [d for d in os.listdir(".") if d.startswith("ffmpeg-") and d.endswith("-static")]
    if st: os.environ["PATH"] = os.path.abspath(st[0]) + ":" + os.environ["PATH"]
    enc = subprocess.run(["ffmpeg","-encoders"], capture_output=True, text=True).stdout
assert "libopencore_amrnb" in enc, "AMR-NB encoder unavailable; enable Internet."
for pkg in ["librosa","soundfile","scikit-learn","scipy","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import librosa, soundfile as sf
print("AMR-NB encoder available. Dependencies installed.")

AMR-NB encoder available. Dependencies installed.


In [3]:
# Cell 3 - Discover the three datasets
def find_file(pattern):
    hits = list(IN.rglob(pattern)); return hits[0] if hits else None

# UrbanSound8K (10-fold, 10 classes)
us8k_csv = find_file("UrbanSound8K.csv")
assert us8k_csv, "UrbanSound8K.csv not found."
us8k_base = us8k_csv.parent
if not (us8k_base/"audio").exists():
    for c in [us8k_csv.parent.parent, us8k_csv.parent.parent.parent]:
        if (c/"audio").exists(): us8k_base = c; break
us8k_official = (us8k_base/"audio"/"fold1").exists()
US8K = []
with open(us8k_csv) as fh:
    for r in csv.DictReader(fh):
        fold=r["fold"]; sub=f"audio/fold{fold}" if us8k_official else f"fold{fold}"
        US8K.append((str(us8k_base/sub/r["slice_file_name"]), r["class"], int(r["fold"])))
US8K_CLASSES = sorted(set(c for _,c,_ in US8K))
print(f"UrbanSound8K : {len(US8K)} clips, {len(US8K_CLASSES)} classes, 10 folds")

# ESC-50 (5-fold, 50 classes)
esc_csv = find_file("esc50.csv")
ESC = []
if esc_csv:
    with open(esc_csv) as fh: first = next(csv.DictReader(fh))["filename"]
    esc_audio = find_file(first)
    if esc_audio:
        esc_dir = esc_audio.parent
        with open(esc_csv) as fh:
            for r in csv.DictReader(fh):
                ESC.append((str(esc_dir/r["filename"]), r["category"], int(r["fold"])))
ESC_CLASSES = sorted(set(c for _,c,_ in ESC))
print(f"ESC-50       : {len(ESC)} clips, {len(ESC_CLASSES)} classes, 5 folds")

# Speech Commands (keyword classification, assign 5 folds deterministically)
KEYWORDS = ["yes","no","up","down","left","right","on","off","stop","go"]
SC = []; sc_root = None
yes_dir = find_file("yes")
if yes_dir and yes_dir.is_dir(): sc_root = yes_dir.parent
if sc_root:
    for kw in KEYWORDS:
        d = sc_root/kw
        if not d.exists(): continue
        files = sorted(d.glob("*.wav"))[:300]   # cap per keyword for balance
        for i, f in enumerate(files):
            SC.append((str(f), kw, 1 + (i % 5)))   # 5 folds within each keyword
SC_CLASSES = sorted(set(c for _,c,_ in SC))
print(f"SpeechCmds   : {len(SC)} clips, {len(SC_CLASSES)} classes, 5 folds")

assert len(US8K)==8732, "UrbanSound8K count unexpected"
assert len(ESC)>0, "ESC-50 not found -- required for Experiment 3"
assert len(SC)>0, "Speech Commands not found -- required for Experiment 1"
print("\nAll three datasets located.")

UrbanSound8K : 8732 clips, 10 classes, 10 folds
ESC-50       : 2000 clips, 50 classes, 5 folds
SpeechCmds   : 3000 clips, 10 classes, 5 folds

All three datasets located.


## Shared infrastructure: codec, features, model, training

In [4]:
# Cell 4 - AMR-NB degradation, mel features, cache, model, training
import librosa, soundfile as sf
from tqdm.auto import tqdm
from multiprocessing.pool import ThreadPool
import multiprocessing as mp

def amr_nb(inp, out, kbps=AMR_KBPS):
    stem = str(out) + ".amr"
    e = subprocess.run(["ffmpeg","-y","-i",str(inp),"-ac","1","-ar","8000",
                        "-b:a",kbps,"-c:a","libopencore_amrnb","-f","amr",stem],
                       capture_output=True, timeout=30)
    if e.returncode != 0: raise RuntimeError(f"encode {Path(inp).name}")
    d = subprocess.run(["ffmpeg","-y","-i",stem,"-ar",str(TARGET_SR),"-ac","1",str(out)],
                       capture_output=True, timeout=30)
    if Path(stem).exists(): os.remove(stem)
    if d.returncode != 0: raise RuntimeError(f"decode {Path(inp).name}")

def degrade_corpus(rows, name):
    """AMR-NB degrade a corpus into PROC/<name>_amr/, resumable."""
    outdir = PROC / f"{name}_amr"
    tasks = []
    for src, _, fold in rows:
        fd = outdir / f"fold{fold}"; fd.mkdir(parents=True, exist_ok=True)
        tasks.append((src, str(fd / Path(src).name)))
    todo = [(s,d) for s,d in tasks if not Path(d).exists()]
    if not todo:
        print(f"  {name}: AMR corpus cached"); return
    def _one(t):
        s,d=t
        try: amr_nb(s,d); return True
        except Exception: return False
    with ThreadPool(max(1,mp.cpu_count()-1)) as pool:
        list(tqdm(pool.imap(_one, todo, chunksize=32), total=len(todo), desc=f"{name} AMR"))
    print(f"  {name}: {len(list(outdir.rglob(chr(42)+chr(46)+chr(119)+chr(97)+chr(118)))) } AMR clips")

def amr_path(name, fold, src):
    return str(PROC / f"{name}_amr" / f"fold{fold}" / Path(src).name)

# ---- mel features + cache ----
def to_mel(path):
    y,_ = librosa.load(path, sr=TARGET_SR, mono=True, duration=MAX_DUR)
    L = int(TARGET_SR*MAX_DUR); y = np.pad(y,(0,max(0,L-len(y))))[:L]
    m = librosa.power_to_db(librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP), ref=np.max)
    return ((m-m.mean())/(m.std()+1e-6)).astype(np.float32)

def mel_path(condition, fold, name):
    d = MEL / condition / f"fold{fold}"; d.mkdir(parents=True, exist_ok=True)
    return d / (name + ".npy")

def cache_mels(condition, rows, source_fn):
    """source_fn(fold, src) -> audio path to read."""
    todo = []
    for src, cls, fold in rows:
        cp = mel_path(condition, fold, Path(src).name)
        if cp.exists(): continue
        todo.append((source_fn(fold, src), str(cp)))
    if not todo:
        print(f"  mel {condition}: cached"); return
    def _one(t):
        ap, cp = t
        try: np.save(cp, to_mel(ap)); return True
        except Exception: return False
    with ThreadPool(max(1,mp.cpu_count()-1)) as pool:
        list(tqdm(pool.imap(_one, todo, chunksize=32), total=len(todo), desc=f"mel {condition}"))
    print(f"  mel {condition}: cached {len(todo)}")

# ---- SpecAugment for the generic-augmentation baseline (applied to cached mel) ----
def spec_augment(mel, n_freq=2, n_time=2, F=15, T=25, rng=None):
    m = mel.copy(); n_mels, n_frames = m.shape
    if rng is None: rng = np.random
    for _ in range(n_freq):
        f = rng.randint(0, F+1); f0 = rng.randint(0, max(1, n_mels-f)); m[f0:f0+f,:] = m.min()
    for _ in range(n_time):
        t = rng.randint(0, T+1); t0 = rng.randint(0, max(1, n_frames-t)); m[:,t0:t0+t] = m.min()
    return m

def add_noise_mel(path, snr_db, rng):
    """Load audio, add Gaussian noise at given SNR, return mel."""
    y,_ = librosa.load(path, sr=TARGET_SR, mono=True, duration=MAX_DUR)
    L=int(TARGET_SR*MAX_DUR); y=np.pad(y,(0,max(0,L-len(y))))[:L]
    p_sig = np.mean(y**2) + 1e-12
    p_noise = p_sig / (10**(snr_db/10))
    y = y + rng.normal(0, np.sqrt(p_noise), size=y.shape).astype(np.float32)
    m = librosa.power_to_db(librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP), ref=np.max)
    return ((m-m.mean())/(m.std()+1e-6)).astype(np.float32)

# ---- datasets ----
class CachedMel(Dataset):
    def __init__(self, files, labels): self.files=files; self.labels=labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return torch.tensor(np.load(self.files[i])).unsqueeze(0), self.labels[i]

def split_from_cache(condition, rows, folds, classes):
    ci={c:i for i,c in enumerate(classes)}; files=[]; labels=[]
    for src, cls, fold in rows:
        if fold not in folds: continue
        cp = mel_path(condition, fold, Path(src).name)
        if not cp.exists(): continue
        files.append(str(cp)); labels.append(ci[cls])
    return CachedMel(files, labels)

class InMemoryMel(Dataset):
    """Holds mel arrays directly in memory (for generic-augmentation training set)."""
    def __init__(self, arrays, labels): self.arrays=arrays; self.labels=labels
    def __len__(self): return len(self.arrays)
    def __getitem__(self, i):
        return torch.tensor(self.arrays[i]).unsqueeze(0), self.labels[i]

# ---- model + training ----
class ResNet50Classifier(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.to_rgb = nn.Conv2d(1,3,1,bias=False)
        b = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        for p in b.parameters(): p.requires_grad=False
        for p in b.layer4.parameters(): p.requires_grad=True
        self.features = nn.Sequential(*list(b.children())[:-1])
        self.embedding = nn.Sequential(nn.Flatten(), nn.Linear(2048,128), nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Linear(128, nc)
    def forward(self, x): return self.classifier(self.embedding(self.features(self.to_rgb(x))))

def train_eval(train_ds, test_ds, nc, seed, tag, return_per_class=False):
    ck = CKPT / f"{tag}.json"
    if ck.exists():
        r=json.load(open(ck)); print(f"  {tag}: cached F1={r['macro_f1']:.3f}"); return r
    torch.manual_seed(seed)
    model = ResNet50Classifier(nc).to(DEVICE)
    opt = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=LR)
    crit = nn.CrossEntropyLoss()
    nv = max(1, len(train_ds)//10); nt = len(train_ds)-nv
    tr, va = torch.utils.data.random_split(train_ds, [nt, nv],
             generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(va, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
    best=float("inf"); pat=0; bst=None
    for _ in range(EPOCHS):
        model.train()
        for X,y in tl:
            X,y=X.to(DEVICE),y.to(DEVICE); opt.zero_grad(); crit(model(X),y).backward(); opt.step()
        model.eval(); vloss=0.0
        with torch.no_grad():
            for X,y in vl: vloss += crit(model(X.to(DEVICE)), y.to(DEVICE)).item()
        vloss/=len(vl)
        if vloss<best: best=vloss; pat=0; bst={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=PATIENCE: break
    model.load_state_dict(bst); model.eval()
    pred=[]; true=[]
    for X,y in DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True):
        pred += model(X.to(DEVICE)).argmax(1).cpu().tolist(); true += y.tolist()
    r = {"macro_f1": float(f1_score(true,pred,average="macro",zero_division=0))}
    if return_per_class:
        r["per_class"] = f1_score(true,pred,average=None,zero_division=0).tolist()
    json.dump(r, open(ck,"w"))
    del model,bst; torch.cuda.empty_cache()
    print(f"  {tag}: F1={r['macro_f1']:.3f}")
    return r

print("Shared infrastructure ready.")

Shared infrastructure ready.


## Experiment 2 - Codec-agnostic augmentation baseline
A model trained with generic augmentation (additive Gaussian noise at 10--20 dB SNR and spectrogram masking, no codec) is tested on AMR-NB 4.75 kbit/s UrbanSound8K. Its recovery is compared against the codec-aware augmentation result from the main experiment (0.698).

In [6]:
# Cell 6 - Experiment 2: generic-augmentation baseline on UrbanSound8K
print("Experiment 2: degrading UrbanSound8K through AMR-NB ...")
degrade_corpus(US8K, "us8k")
print("Caching UrbanSound8K clean and AMR mels ...")
cache_mels("us8k_clean", US8K, lambda fold, src: src)
cache_mels("us8k_amr",   US8K, lambda fold, src: amr_path("us8k", fold, src))

us8k_folds = sorted(set(f for _,_,f in US8K))
nc_us = len(US8K_CLASSES)

def build_generic_aug_trainset(train_folds, seed):
    """Clean mels + noise-augmented mels + SpecAugment copies; built in memory."""
    rng = np.random.RandomState(seed)
    ci = {c:i for i,c in enumerate(US8K_CLASSES)}
    arrays=[]; labels=[]
    for src, cls, fold in US8K:
        if fold not in train_folds: continue
        base = np.load(mel_path("us8k_clean", fold, Path(src).name))
        arrays.append(base); labels.append(ci[cls])                 # clean
        snr = rng.uniform(10, 20)
        arrays.append(add_noise_mel(src, snr, rng)); labels.append(ci[cls])  # noise
        arrays.append(spec_augment(base, rng=rng)); labels.append(ci[cls])   # SpecAugment
    return InMemoryMel(arrays, labels)

print("\nUrbanSound8K: generic-augmentation training, tested on AMR-NB 4.75k")
scores=[]
for te in us8k_folds:
    tr=[f for f in us8k_folds if f!=te]
    tag = f"exp2_genericaug_f{te}"
    ck = CKPT / f"{tag}.json"
    if ck.exists():
        r=json.load(open(ck)); print(f"  {tag}: cached F1={r['macro_f1']:.3f}"); scores.append(r["macro_f1"]); continue
    train_ds = build_generic_aug_trainset(tr, SEED+te)
    test_ds  = split_from_cache("us8k_amr", US8K, [te], US8K_CLASSES)
    r = train_eval(train_ds, test_ds, nc_us, SEED, tag)
    scores.append(r["macro_f1"])

exp2 = {"generic_aug": {"mean": float(np.mean(scores)), "std": float(np.std(scores)), "f1all": scores},
        "codec_aug_reference": REF_US8K_AUG,
        "clean_to_amr_reference": REF_US8K_AMR,
        "clean_reference": REF_US8K_CLEAN}
json.dump(exp2, open(RESULTS/"experiment2_generic_aug.json","w"), indent=2)
gen = exp2["generic_aug"]["mean"]
rec_gen = 100*(gen - REF_US8K_AMR)/(REF_US8K_CLEAN - REF_US8K_AMR)
rec_codec = 100*(REF_US8K_AUG - REF_US8K_AMR)/(REF_US8K_CLEAN - REF_US8K_AMR)
print("\n=== Experiment 2 result ===")
print(f"  Generic augmentation : {gen:.3f}  (recovery {rec_gen:.0f}%)")
print(f"  Codec-aware aug      : {REF_US8K_AUG:.3f}  (recovery {rec_codec:.0f}%)")
print(f"  -> Codec-aware augmentation is {'BETTER' if REF_US8K_AUG > gen else 'NOT better'} "
      f"than generic augmentation by {(REF_US8K_AUG-gen)*100:.1f} pp")

Experiment 2: degrading UrbanSound8K through AMR-NB ...


us8k AMR:   0%|          | 0/8732 [00:00<?, ?it/s]

  us8k: 8732 AMR clips
Caching UrbanSound8K clean and AMR mels ...


mel us8k_clean:   0%|          | 0/8732 [00:00<?, ?it/s]

  mel us8k_clean: cached 8732


mel us8k_amr:   0%|          | 0/8732 [00:00<?, ?it/s]

  mel us8k_amr: cached 8732

UrbanSound8K: generic-augmentation training, tested on AMR-NB 4.75k
  exp2_genericaug_f1: F1=0.360
  exp2_genericaug_f2: F1=0.360
  exp2_genericaug_f3: F1=0.387
  exp2_genericaug_f4: F1=0.357
  exp2_genericaug_f5: F1=0.435
  exp2_genericaug_f6: F1=0.476
  exp2_genericaug_f7: F1=0.382
  exp2_genericaug_f8: F1=0.361
  exp2_genericaug_f9: F1=0.474
  exp2_genericaug_f10: F1=0.384

=== Experiment 2 result ===
  Generic augmentation : 0.398  (recovery 7%)
  Codec-aware aug      : 0.698  (recovery 79%)
  -> Codec-aware augmentation is BETTER than generic augmentation by 30.0 pp


## Experiment 3 - ESC-50 generalisation
The full clean, matched, and augmented pipeline is repeated on ESC-50 (50 classes, five folds). This tests whether the findings generalise beyond UrbanSound8K and, with 50 classes, supports a per-class frequency-dependence analysis.

In [7]:
# Cell 7 - Experiment 3: ESC-50 end-to-end
print("Experiment 3: degrading ESC-50 through AMR-NB ...")
degrade_corpus(ESC, "esc")
print("Caching ESC-50 clean and AMR mels ...")
cache_mels("esc_clean", ESC, lambda fold, src: src)
cache_mels("esc_amr",   ESC, lambda fold, src: amr_path("esc", fold, src))

esc_folds = sorted(set(f for _,_,f in ESC))
nc_esc = len(ESC_CLASSES)
exp3 = {}

def build_esc_aug_trainset(train_folds):
    ci={c:i for i,c in enumerate(ESC_CLASSES)}; files=[]; labels=[]
    for src, cls, fold in ESC:
        if fold not in train_folds: continue
        for cond in ["esc_clean","esc_amr"]:
            cp = mel_path(cond, fold, Path(src).name)
            if cp.exists(): files.append(str(cp)); labels.append(ci[cls])
    return CachedMel(files, labels)

# clean->clean, clean->amr, matched, augmented
for label, train_cond, test_cond in [("clean_to_clean","esc_clean","esc_clean"),
                                      ("clean_to_amr","esc_clean","esc_amr"),
                                      ("matched","esc_amr","esc_amr")]:
    print(f"\nESC-50: {label}")
    scores=[]; pcs=[]
    for te in esc_folds:
        tr=[f for f in esc_folds if f!=te]
        train_ds = split_from_cache(train_cond, ESC, tr, ESC_CLASSES)
        test_ds  = split_from_cache(test_cond, ESC, [te], ESC_CLASSES)
        r = train_eval(train_ds, test_ds, nc_esc, SEED, f"exp3_esc_{label}_f{te}",
                       return_per_class=(label=="clean_to_amr" or label=="clean_to_clean"))
        scores.append(r["macro_f1"])
        if "per_class" in r: pcs.append(r["per_class"])
    exp3[label] = {"mean": float(np.mean(scores)), "std": float(np.std(scores)), "f1all": scores}
    if pcs: exp3[label]["per_class_mean"] = np.mean(pcs, axis=0).tolist()

# augmented (train on clean+amr union)
print("\nESC-50: augmented (clean + AMR)")
scores=[]
for te in esc_folds:
    tr=[f for f in esc_folds if f!=te]
    tag=f"exp3_esc_aug_f{te}"; ck=CKPT/f"{tag}.json"
    if ck.exists():
        r=json.load(open(ck)); print(f"  {tag}: cached F1={r['macro_f1']:.3f}"); scores.append(r["macro_f1"]); continue
    train_ds = build_esc_aug_trainset(tr)
    test_ds  = split_from_cache("esc_amr", ESC, [te], ESC_CLASSES)
    r = train_eval(train_ds, test_ds, nc_esc, SEED, tag)
    scores.append(r["macro_f1"])
exp3["augmented"] = {"mean": float(np.mean(scores)), "std": float(np.std(scores)), "f1all": scores}

json.dump(exp3, open(RESULTS/"experiment3_esc50.json","w"), indent=2)
A=exp3["clean_to_clean"]["mean"]; C=exp3["clean_to_amr"]["mean"]
M=exp3["matched"]["mean"]; AUG=exp3["augmented"]["mean"]
print("\n=== Experiment 3 result (ESC-50) ===")
print(f"  clean->clean : {A:.3f}")
print(f"  clean->AMR   : {C:.3f}  (drop {(A-C)*100:.1f} pp)")
print(f"  matched      : {M:.3f}  (recovery {100*(M-C)/(A-C+1e-9):.0f}%)")
print(f"  augmented    : {AUG:.3f}  (recovery {100*(AUG-C)/(A-C+1e-9):.0f}%)")
print(f"  UrbanSound8K reference drop: {(REF_US8K_CLEAN-REF_US8K_AMR)*100:.1f} pp")

Experiment 3: degrading ESC-50 through AMR-NB ...


esc AMR:   0%|          | 0/2000 [00:00<?, ?it/s]

  esc: 2000 AMR clips
Caching ESC-50 clean and AMR mels ...


mel esc_clean:   0%|          | 0/2000 [00:00<?, ?it/s]

  mel esc_clean: cached 2000


mel esc_amr:   0%|          | 0/2000 [00:00<?, ?it/s]

  mel esc_amr: cached 2000

ESC-50: clean_to_clean
  exp3_esc_clean_to_clean_f1: F1=0.705
  exp3_esc_clean_to_clean_f2: F1=0.713
  exp3_esc_clean_to_clean_f3: F1=0.741
  exp3_esc_clean_to_clean_f4: F1=0.767
  exp3_esc_clean_to_clean_f5: F1=0.681

ESC-50: clean_to_amr
  exp3_esc_clean_to_amr_f1: F1=0.253
  exp3_esc_clean_to_amr_f2: F1=0.297
  exp3_esc_clean_to_amr_f3: F1=0.318
  exp3_esc_clean_to_amr_f4: F1=0.349
  exp3_esc_clean_to_amr_f5: F1=0.309

ESC-50: matched
  exp3_esc_matched_f1: F1=0.666
  exp3_esc_matched_f2: F1=0.623
  exp3_esc_matched_f3: F1=0.643
  exp3_esc_matched_f4: F1=0.620
  exp3_esc_matched_f5: F1=0.595

ESC-50: augmented (clean + AMR)
  exp3_esc_aug_f1: F1=0.664
  exp3_esc_aug_f2: F1=0.655
  exp3_esc_aug_f3: F1=0.617
  exp3_esc_aug_f4: F1=0.701
  exp3_esc_aug_f5: F1=0.586

=== Experiment 3 result (ESC-50) ===
  clean->clean : 0.721
  clean->AMR   : 0.305  (drop 41.6 pp)
  matched      : 0.629  (recovery 78%)
  augmented    : 0.645  (recovery 82%)
  UrbanSound8K refe

In [8]:
# Cell 8 - ESC-50 per-class frequency-dependence (n=50 classes)
import librosa
from scipy.stats import spearmanr

if "per_class_mean" in exp3.get("clean_to_clean",{}) and "per_class_mean" in exp3.get("clean_to_amr",{}):
    clean_pc = np.array(exp3["clean_to_clean"]["per_class_mean"])
    amr_pc   = np.array(exp3["clean_to_amr"]["per_class_mean"])
    drop_pc  = clean_pc - amr_pc
    # high-frequency energy fraction per class (clean audio)
    def hf_fraction(path, cut=3400):
        y,_=librosa.load(path, sr=TARGET_SR, duration=MAX_DUR)
        S=np.abs(librosa.stft(y))**2; f=librosa.fft_frequencies(sr=TARGET_SR)
        return float(S[f>=cut].sum()/(S.sum()+1e-12))
    from collections import defaultdict
    hf_by_class = defaultdict(list)
    for src, cls, fold in ESC:
        hf_by_class[cls].append(src)
    Hc = []
    for cls in ESC_CLASSES:
        paths = hf_by_class[cls][:10]   # sample up to 10 clips per class
        Hc.append(np.mean([hf_fraction(p) for p in paths]))
    Hc = np.array(Hc)
    rho, p = spearmanr(Hc, drop_pc)
    freq_dep = {"spearman_rho": float(rho), "spearman_p": float(p), "n_classes": len(ESC_CLASSES),
                "per_class_drop": {c: float(d) for c,d in zip(ESC_CLASSES, drop_pc)},
                "per_class_hf": {c: float(h) for c,h in zip(ESC_CLASSES, Hc)}}
    json.dump(freq_dep, open(RESULTS/"experiment3_frequency_dependence.json","w"), indent=2)
    print(f"ESC-50 frequency-dependence (n={len(ESC_CLASSES)} classes):")
    print(f"  Spearman rho = {rho:+.3f}, p = {p:.4f}")
    print(f"  (Main experiment on UrbanSound8K: rho=+0.26, p=0.47 at n=10)")
    top = sorted(zip(ESC_CLASSES, drop_pc), key=lambda x:-x[1])[:5]
    print(f"  Most vulnerable classes: {[c for c,_ in top]}")
else:
    print("Per-class data not available; ensure Experiment 3 completed.")

ESC-50 frequency-dependence (n=50 classes):
  Spearman rho = +0.402, p = 0.0038
  (Main experiment on UrbanSound8K: rho=+0.26, p=0.47 at n=10)
  Most vulnerable classes: ['church_bells', 'brushing_teeth', 'crying_baby', 'train', 'pouring_water']


In [9]:
# Cell 9 - Bundle all supporting-experiment results
import shutil
summary = {}
for f in ["experiment1_speech_vs_env.json","experiment2_generic_aug.json",
          "experiment3_esc50.json","experiment3_frequency_dependence.json"]:
    p = RESULTS/f
    if p.exists(): summary[f.replace(".json","")] = json.load(open(p))
json.dump(summary, open(RESULTS/"supporting_experiments_summary.json","w"), indent=2)
shutil.make_archive(str(WORK/"supporting_experiments_results"),"zip",str(RESULTS))
print("Download supporting_experiments_results.zip from the Output tab.")
print("Files:", sorted(p.name for p in RESULTS.glob("*.json")))

Download supporting_experiments_results.zip from the Output tab.
Files: ['experiment1_speech_vs_env.json', 'experiment2_generic_aug.json', 'experiment3_esc50.json', 'experiment3_frequency_dependence.json', 'supporting_experiments_summary.json']
